In [6]:
# ============================================================
# Seed Coat Annotation Plot
# GeneSet DotPlot + Marker DotPlot
# ============================================================

library(Seurat)
library(dplyr)
library(ggplot2)

# ============================================================
# 🙌 参数
# ============================================================

prefix <- "bent_seedcoat"# 🙌

rds_file <- "/data/work/RNA_AraSeed/rds/bent/bent_recluster_res1.2.rds"# 🙌

marker_file <- "/data/work/00markers/Cell_types_markerGenes/bentseedcoat.txt"# 🙌

cluster_col <- "seurat_clusters"# 🙌
setwd("/data/work/RNA_AraSeed/CellTypeAnno/test")# 🙌
# ============================================================
# 读取Seurat对象
# ============================================================

obj <- readRDS(rds_file)

Idents(obj) <- cluster_col

# ============================================================
# 读取marker
# ============================================================

marker_df <- read.delim(
  marker_file,
  header = TRUE,
  stringsAsFactors = FALSE
)

# ============================================================
# 整理marker
# ============================================================

marker_df$Gene_ID <- toupper(trimws(marker_df$Gene_ID))
marker_df$Cell_type <- trimws(marker_df$Cell_type)

marker_df <- marker_df %>%
  distinct(
    Gene_ID,
    Cell_type,
    .keep_all = TRUE
  )

# ============================================================
# 保留存在基因
# ============================================================

marker_df <- marker_df %>%
  filter(
    Gene_ID %in% rownames(obj)
  )

cat(
  "Genes retained:",
  nrow(marker_df),
  "\n"
)

write.csv(
  marker_df,
  paste0(prefix, "_Markers_Used.csv"),
  row.names = FALSE
)

# ============================================================
# Gene Sets
# ============================================================

gene_sets <- split(
  marker_df$Gene_ID,
  marker_df$Cell_type
)

cat("\nGene Set Size:\n")

print(
  sapply(
    gene_sets,
    length
  )
)

# ============================================================
# Figure1
# GeneSet Score
# ============================================================

obj <- AddModuleScore(
  object = obj,
  features = gene_sets,
  name = "GeneSet_"
)

score_cols <- grep(
  "^GeneSet_",
  colnames(obj@meta.data),
  value = TRUE
)

score_map <- data.frame(
  ScoreCol = score_cols,
  CellType = names(gene_sets)
)

# ============================================================
# GeneSet DotPlot 数据
# ============================================================

plot_list <- list()

for(i in seq_along(score_cols)){

  score_col <- score_cols[i]

  tmp <- obj@meta.data %>%
    group_by(
      .data[[cluster_col]]
    ) %>%
    summarise(
      AvgScore = mean(
        .data[[score_col]]
      ),
      PctHigh = mean(
        .data[[score_col]] > 0
      ) * 100
    )

  tmp$GeneSet <- names(gene_sets)[i]

  colnames(tmp)[1] <- "Cluster"

  plot_list[[i]] <- tmp
}

geneSet_df <- do.call(
  rbind,
  plot_list
)

# ============================================================
# GeneSet DotPlot
# ============================================================

p1 <- ggplot(
  geneSet_df,
  aes(
    x = GeneSet,
    y = factor(Cluster)
  )
) +
  geom_point(
    aes(
      size = PctHigh,
      color = AvgScore
    )
  ) +
  scale_color_gradient2(
    low = "#2166AC",
    mid = "white",
    high = "#B2182B"
  ) +
  theme_bw() +
  labs(
    title = paste0(
      prefix,
      " Gene Set DotPlot"
    ),
    x = "",
    y = "Cluster"
  ) +
  theme(
    axis.text.x = element_text(
      angle = 45,
      hjust = 1,
      face = "bold"
    )
  )

ggsave(
  paste0(
    prefix,
    "_GeneSet_DotPlot.pdf"
  ),
  p1,
  width = 8,
  height = 8
)



# ============================================================
# Figure2
# Marker DotPlot
# ============================================================

feature_list <- split(
  marker_df$Gene_ID,
  marker_df$Cell_type
)

p2 <- DotPlot(
  object = obj,
  features = feature_list,
  group.by = cluster_col,
  dot.scale = 8
) +
  scale_color_gradientn(
    colours = c(
      "#2166AC",
      "#67A9CF",
      "#D1E5F0",
      "#F7F7F7",
      "#FDDBC7",
      "#EF8A62",
      "#B2182B"
    )
  ) +
  RotatedAxis() +
  theme_bw() +
  theme(
    axis.text.x = element_text(
      angle = 90,
      hjust = 1,
      size = 7
    ),
    axis.text.y = element_text(
      size = 8
    ),
    strip.text = element_text(
      face = "bold",
      size = 10
    )
  ) +
  labs(
    title = paste0(
      prefix,
      " Marker DotPlot"
    )
  )

ggsave(
  paste0(
    prefix,
    "_Marker_DotPlot.pdf"
  ),
  p2,
  width = 18,
  height = 10
)



# ============================================================
# 保存GeneSet统计
# ============================================================

write.csv(
  geneSet_df,
  paste0(
    prefix,
    "_GeneSet_DotPlot_Data.csv"
  ),
  row.names = FALSE
)

cat("\nAll Finished!\n")

Genes retained: 18 

Gene Set Size:
                CZSC                   ii                   oi 
                   7                    3                    4 
SeedCoat_endothelium 
                   4 


Scale for colour is already present.
Adding another scale for colour, which will replace the existing scale.



All Finished!
